# Lesson 14: Agent evaluation (answer + tool-call trajectory)

ToyAIKit agent on Groq (same setup as Module 1 lesson 15), **8-question sample** (free tier). Course full results: 50 agent answers in `data/course/`.

In [1]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

# Course: openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/course/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)
len(documents)

161

In [4]:
doc_idx = {}
for doc in documents:
    doc_idx[doc["id"]] = doc
ground_truth = [r for r in ground_truth if r["document"] in doc_idx]

In [5]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [6]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.pricing import PricingConfig
from evaluation_utils import MODEL, INPUT_PRICE_PER_MILLION, OUTPUT_PRICE_PER_MILLION

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

pricing = PricingConfig()
pricing.register_model(MODEL, input_price=INPUT_PRICE_PER_MILLION, output_price=OUTPUT_PRICE_PER_MILLION)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model=MODEL, client=openai_client),   # course: OpenAIClient(model="gpt-5.4-mini")
    pricing_config=pricing,
)

In [7]:
def extract_tool_calls(messages):
    tool_calls = []
    for message in messages:
        if isinstance(message, dict):
            continue
        if message.type == "function_call":
            tool_calls.append({"name": message.name, "arguments": message.arguments})
    return tool_calls

In [8]:
from evaluation_utils import with_retry

def generate_agent_answer(rec):
    doc_id = rec["document"]
    result = with_retry(lambda: runner.loop(prompt=rec["question"]))
    tool_calls = extract_tool_calls(result.all_messages)
    return {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": doc_idx[doc_id]["answer"],
        "tool_calls": json.dumps(tool_calls),
        "cost": float(result.cost.total_cost),
        "document": doc_id,
    }

r = generate_agent_answer(ground_truth[0])
print(r["question"]); print(r["tool_calls"]); print(r["answer_agent"][:400])

Is it okay to join the course late if I just found it now?
[{"name": "search", "arguments": "{\"query\":\"late joining course join late policy\"}"}]
Yes—you can still join the course even if you just discovered it. The only catch is that, to earn a certificate, you’ll need to submit your Capstone project before the final submission window closes. Apart from that, you’re free to start watching the videos, work through the notebooks, and do the homework at any time. Happy learning!


In [9]:
import random, time
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

random.seed(7)
sample = random.sample(ground_truth, 8)
t0 = time.time()
with ThreadPoolExecutor(max_workers=2) as pool:
    agent_answers = map_progress(pool, sample, generate_agent_answer)
print(f"{time.time() - t0:.0f} s")

df_agent = pd.DataFrame(agent_answers)
df_agent.to_csv("data/agent-answers-groq-sample.csv", index=False)
print("agent cost $", round(df_agent["cost"].sum(), 6))
df_agent["n_tool_calls"] = df_agent["tool_calls"].map(lambda s: len(json.loads(s)))
df_agent[["question", "n_tool_calls"]]

  0%|          | 0/8 [00:00<?, ?it/s]

212 s
agent cost $ 0.006066


,question,n_tool_calls
0,Why am I getting `JSONDecodeError: Expecting v...,1
1,How can I point Python to the right .env path ...,1
2,How can I find a column’s percentile value in ...,1
3,Is peer-review of the capstone project require...,1
4,"Will there be another run of this course, and ...",4
5,How do I make the project easier and cheaper f...,1
6,What do I need to write in .wslconfig to set W...,1
7,What should I change in my .env file if postgr...,1


## Judging answers and trajectories

In [10]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(description="Reasoning about whether the final answer is correct.")
    answer_score: Literal["good", "bad"] = Field(description="'good' if the final answer matches the original answer.")
    trajectory_reasoning: str = Field(description="Reasoning about whether the tool calls were useful.")
    trajectory_score: Literal["good", "bad"] = Field(description="'good' if the tool calls were reasonable for the question.")

In [11]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [12]:
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec):
    tool_calls = rec["tool_calls"]
    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)
    prompt = agent_judge_prompt.format(question=rec["question"], answer_orig=rec["answer_orig"],
                                       answer_agent=rec["answer_agent"], tool_calls=json.dumps(tool_calls, indent=2))
    return llm_structured_retry(openai_client, agent_judge_instructions, prompt, AgentEvaluation)

def judge_agent_record(rec):
    ev, usage = evaluate_agent_answer(rec)
    return {"question": rec["question"], "document": rec["document"],
            "answer_score": ev.answer_score, "answer_reasoning": ev.answer_reasoning,
            "trajectory_score": ev.trajectory_score, "trajectory_reasoning": ev.trajectory_reasoning}, usage

with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

df_agent_eval = pd.DataFrame([e for e, u in results])
print("judge cost $", round(calc_total_price([u for e, u in results]), 6))
print(df_agent_eval["answer_score"].value_counts().to_dict(), df_agent_eval["trajectory_score"].value_counts().to_dict())
df_agent_eval.to_csv("data/agent-evaluations-groq-sample.csv", index=False)

  0%|          | 0/8 [00:00<?, ?it/s]

judge cost $ 0.00297
{'good': 8} {'good': 7, 'bad': 1}


In [13]:
df_agent_eval[(df_agent_eval["answer_score"] == "bad") | (df_agent_eval["trajectory_score"] == "bad")][["question", "answer_reasoning", "trajectory_reasoning"]]

,question,answer_reasoning,trajectory_reasoning
4,"Will there be another run of this course, and ...",The agent’s response includes all key points f...,"The agent made four separate search calls, all..."


## Course full results (50 agent answers, gpt-5.4-mini)

In [14]:
df_course = pd.read_csv("data/course/agent-evaluations.csv")
print(df_course["answer_score"].value_counts().to_dict(), df_course["trajectory_score"].value_counts().to_dict())

{'good': 45, 'bad': 5} {'good': 49, 'bad': 1}
